<a href="https://colab.research.google.com/github/tanmayidarbha/1MS25IS126-assignment2/blob/main/agent_loop_demo(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [5]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [6]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
ENTERTAINMENT = {
    "manga": (
        "Omniscient reader's viewpoint is a manga inspired from a korean web novel. It beautifully depicts the mc's connection with the novel's characters in the story ."
        "The greatest estate developer is a manga revolved mostly around a modern-day civil engineering student who finds himself in a novel he read. He brings a lot of mind-blowing changes in the debt-ridden estate and saves it."
        "Wind Breaker manga revolves around a character who only knows to study and cycle. He makes new friends and joins a tournament. What started as a fun hobby ends up enlightening him about his uncle's past"
        "School bus graveyard is a thrilling webtoon that depicts a high-schooler who has a weird connection with beings from a different dimensions. She and her new group of friends must find ways to survive each night as they find themselves thrown into the phantom dimension."
    ),
    "novels":(

        "A good girl's guide to murder is a murder-thriller in which a student who simply takes on a case for her school project, ends up finding real clues about the missing people. She along with her friend solve the murder case and bring peace to their small town."
        "Never lie is a thriller by freida in which a couple gets stranded in a creepy house belonging to a dead person during house-hunting. Turns out both of them have secrets they don't want the other to know."
        "The empyrean series revolves around a fragile girl who has a rather strong heart and mind. She goes through a lot of things from bonding dragons to losing loved ones during her journey in the war college. "
    ),

    "shows/dramas":(
        "Nightmare teacher is a kdrama is a thriller which is about a teacher who knows more than he is letting on."
        "TwentyfiveTwentyone is a high school kdrama about fencing. Two friend end up on the opposing teams with a lot of weight on their shoulders. We still see them grow and navigate their love lives."
        "100 days of deception is about a girl who doesnt want anything to do with the war going on around her and just wants to go to america. She ends up working as a spy for korean independence and starts spying on the japanese."
    ),

}

In [7]:
# @title Define a tool
def search_entertainment(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in ENTERTAINMENT.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant information found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_entertainment("minimum of 4"))

[Source: shows/dramas]
Nightmare teacher is a kdrama is a thriller which is about a teacher who knows more than he is letting on.TwentyfiveTwentyone is a high school kdrama about fencing. Two friend end up on the opposing teams with a lot of weight on their shoulders. We still see them grow and navigate their love lives.100 days of deception is about a girl who doesnt want anything to do with the war going on around her and just wants to go to america. She ends up working as a spy for korean independence and starts spying on the japanese.

[Source: novels]
A good girl's guide to murder is a murder-thriller in which a student who simply takes on a case for her school project, ends up finding real clues about the missing people. She along with her friend solve the murder case and bring peace to their small town.Never lie is a thriller by freida in which a couple gets stranded in a creepy house belonging to a dead person during house-hunting. Turns out both of them have secrets they don't

In [8]:
# @title Ask gemini to search for something
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    # This will break.
    # TODO: Find the right model to call in this cell from developer docs.
    model_name="claude",
    tools=[search_entertainment],
    system_instruction=(
        '''
        You are a professional reviewer for various entertainment shows, novels and comics. Recommend me the best sources to read or watch based on my preferences.
        '''
    )
)

print("Model ready:", model.model_name)

Model ready: models/claude


In [11]:

import time
from google import genai
from google.genai import types

# Initialize Gemini client
client = genai.Client()

# Define the entertainment search tool
search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_entertainment",
            description="Search the entertainment records to find the most relevant source.",
            parameters={
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"],
            },
        )
    ]
)

# Configure function calling
config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    ),
)


# Retry API requests if temporary errors occur
def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model="gemini-3.8-flash",
                contents=contents,
                config=config
            )

        except Exception as e:
            error = str(e)

            retryable = any(
                code in error
                for code in ["429", "500", "502", "503", "504"]
            )

            if retryable and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(
                    f"Temporary API error. "
                    f"Retrying in {wait} seconds..."
                )
                time.sleep(wait)
            else:
                raise


# Run the agent
def run_agent(question: str, max_steps: int = 8) -> str:

    print(f"Question: {question}\n" + "-" * 50)

    contents = [
        types.Content(
            role="user",
            parts=[types.Part(text=question)]
        )
    ]

    for step in range(1, max_steps + 1):

        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Preserve the model response and thought signatures
        contents.append(model_content)

        # Check whether the model requested a function call
        calls = [
            p.function_call
            for p in model_content.parts
            if p.function_call
        ]

        # If there are no function calls, print the final answer
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Execute all requested function calls
        result_parts = []

        for call in calls:
            args = dict(call.args)

            print(
                f"Step {step}: model calls "
                f"{call.name}({args})"
            )

            result = search_entertainment(**args)

            print(
                f"  Result preview: {str(result)[:100]}..."
            )

            result_parts.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result}
                )
            )

        # Send all function results back to Gemini
        contents.append(
            types.Content(
                role="user",
                parts=result_parts
            )
        )

    return "(Stopped: hit max_steps)"


# Your question
_ = run_agent("Recommend me engineering related comics")



Question: Recommend me engineering related comics
--------------------------------------------------
Step 1: model calls search_entertainment({'query': 'engineering'})
  Result preview: [Source: manga]
Omniscient reader's viewpoint is a manga inspired from a korean web novel. It beauti...

Final answer:
Here are some of the best engineering-related comics, manga, and graphic novels across different disciplines (civil, aerospace, mechanical, and speculative tech):

---

### 1. Civil & Structural Engineering
* **The Greatest Estate Developer** (*Webtoon / Manhwa*)
  * **Premise:** A modern civil engineering student is reincarnated into a fantasy novel as Lloyd Frontera, the lazy, debt-ridden son of a minor noble. 
  * **Engineering Focus:** He uses real-world principles of civil engineering, surveying, soil mechanics, thermal insulation (ondol floor heating), hydraulics, and bridge design to turn a bankrupt territory into a thriving metropolis. It is hilarious, genuinely clever, and cele

In [12]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "Whats the manga which is based on a charcater cycling and finding about his uncle?",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?",
    "Which drama has a sports components like cycling , fencing etc.?",
    "Suggest me a few thriller novles."
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)


Question: Whats the manga which is based on a charcater cycling and finding about his uncle?
--------------------------------------------------
Step 1: model calls search_entertainment({'query': 'manga cycling uncle'})
  Result preview: [Source: manga]
Omniscient reader's viewpoint is a manga inspired from a korean web novel. It beauti...

Final answer:
The series you are referring to is most likely the popular manhwa/webtoon ***Wind Breaker*** by Jo Yongseok.

### Premise:
* **Protagonist:** **Jay Jo** (Jo Ja-hyun), a high school class president and straight-A student who is pressured heavily by his parents to focus only on his studies.
* **Cycling & Mystery:** Jay loves riding his fixed-gear bicycle and possesses exceptional cycling technique. After being drawn into the underground street-racing scene (the *League of Street* competition) with his newfound crew, *Hummingbird*, he begins to uncover the truth surrounding his late uncle, **Mahon Jo**—a former legendary professional cycl

In [20]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Recommend me a romcom novel for christmas." not in questions)